# Gold — Turismo Bilateral Brasil–Coreia do Sul

## 1. Construção da tabela analítica mensal

Objetivo: construir uma tabela comparativa das chegadas
internacionais entre Brasil e Coreia do Sul.

Período pretendido: 1990–2025.

Fontes:
- chegadas.silver.brasil
- chegadas.silver.coreia

Regras:
- Selecionar os registros relevantes para cada fluxo.
- Preservar as tabelas Silver originais.
- Não misturar totais anuais e registros mensais.
- Verificar a integridade dos indicadores.
- Documentar as diferenças metodológicas entre as fontes.

In [0]:
from pyspark.sql import functions as F

df_brasil = spark.table("chegadas.silver.brasil")
df_coreia = spark.table("chegadas.silver.coreia")

In [0]:
df_coreanos_brasil_mensal = (
    df_brasil
    .filter(
        (F.col("pais") == "República da Coreia") &
        (F.col("ano").between(1998, 2025))
    )
)

print(
    "Registros selecionados:",
    df_coreanos_brasil_mensal.count()
)

In [0]:
meses = {
    "janeiro": 1,
    "fevereiro": 2,
    "março": 3,
    "marco": 3,
    "abril": 4,
    "maio": 5,
    "junho": 6,
    "julho": 7,
    "agosto": 8,
    "setembro": 9,
    "outubro": 10,
    "novembro": 11,
    "dezembro": 12
}

mapa_meses = F.create_map(
    *[
        item
        for nome, numero in meses.items()
        for item in (F.lit(nome), F.lit(numero))
    ]
)

df_coreanos_brasil_mensal = (
    df_coreanos_brasil_mensal
    .withColumn(
        "mes_num",
        F.element_at(
            mapa_meses,
            F.lower(F.trim(F.col("mes")))
        )
    )
)

In [0]:
display(
    df_coreanos_brasil_mensal
    .groupBy("mes", "mes_num")
    .count()
    .orderBy("mes_num", "mes")
)

In [0]:
df_brasil_mensal = (
    df_coreanos_brasil_mensal

    .groupBy("ano", "mes_num")

    .agg(
        F.sum("chegadas").alias("chegadas_observadas"),

        F.count("*").alias("total_registros"),

        F.sum(
            F.when(
                F.col("chegadas").isNull(), 1
            ).otherwise(0)
        ).alias("registros_nulos")
    )

    .withColumnRenamed("mes_num", "mes")

    .withColumn(
        "chegadas",
        F.when(
            F.col("registros_nulos") == 0,
            F.col("chegadas_observadas")
        ).otherwise(F.lit(None).cast("bigint"))
    )

    .withColumn(
        "status_dados",
        F.when(
            F.col("registros_nulos") == 0,
            F.lit("sem_nulos")
        ).otherwise(F.lit("incompleto"))
    )
)

display(
    df_brasil_mensal
    .orderBy("ano", "mes")
)

In [0]:
df_coreia_mensal = (
    df_coreia
    .filter(
        (F.col("categoria_en") == "Brazil") &
        (F.col("granularidade") == "mensal") &
        (F.col("ano").between(1998, 2025))
    )
    .select(
        "ano",
        "mes",
        "chegadas"
    )
)

display(
    df_coreia_mensal
    .orderBy("ano", "mes")
)

print(
    "Registros mensais:",
    df_coreia_mensal.count()
)

In [0]:
display(
    df_coreia_mensal
    .groupBy("ano", "mes")
    .count()
    .filter(F.col("count") != 1)
)

In [0]:
df_coreia_mensal = (
    df_coreia_mensal

    .withColumn(
        "chegadas_observadas",
        F.col("chegadas")
    )

    .withColumn(
        "total_registros",
        F.lit(1)
    )

    .withColumn(
        "registros_nulos",
        F.when(
            F.col("chegadas").isNull(), 1
        ).otherwise(0)
    )

    .withColumn(
        "status_dados",
        F.when(
            F.col("chegadas").isNull(),
            F.lit("incompleto")
        ).otherwise(F.lit("sem_nulos"))
    )
)

In [0]:
# Coreia do Sul → Brasil

df_fluxo_brasil_mensal = (
    df_brasil_mensal
    .withColumn("origem", F.lit("Coreia do Sul"))
    .withColumn("destino", F.lit("Brasil"))
)

# Brasil → Coreia do Sul

df_fluxo_coreia_mensal = (
    df_coreia_mensal
    .withColumn("origem", F.lit("Brasil"))
    .withColumn("destino", F.lit("Coreia do Sul"))
)

# União dos dois fluxos

df_gold_mensal = (
    df_fluxo_brasil_mensal

    .unionByName(df_fluxo_coreia_mensal)

    .select(
        "ano",
        "mes",
        "origem",
        "destino",
        "chegadas",
        "chegadas_observadas",
        "total_registros",
        "registros_nulos",
        "status_dados"
    )

    .orderBy("ano", "mes", "origem")
)

display(df_gold_mensal)

In [0]:
# Quantidade total de registros
print("Total Gold mensal:", df_gold_mensal.count())

# Verificar se existem meses duplicados
display(
    df_gold_mensal
    .groupBy("ano", "mes", "origem", "destino")
    .count()
    .filter(F.col("count") != 1)
)

# Conferir meses com dados incompletos
display(
    df_gold_mensal
    .filter(F.col("status_dados") == "incompleto")
    .orderBy("ano", "mes", "origem")
)

In [0]:
display(
    df_gold_mensal
    .groupBy("ano", "origem", "destino")
    .agg(
        F.countDistinct("mes").alias("meses_disponiveis")
    )
    .filter(F.col("meses_disponiveis") != 12)
    .orderBy("ano", "origem")
)

Identificação de mês ausente

In [0]:
# Criar uma lista com os 12 meses
df_meses = spark.range(1, 13).select(
    F.col("id").cast("int").alias("mes")
)

# Selecionar os meses existentes em 2023
df_meses_2023 = (
    df_brasil_mensal
    .filter(F.col("ano") == 2023)
    .select("mes")
    .distinct()
)

# Identificar qual mês não aparece
display(
    df_meses.join(
        df_meses_2023,
        on="mes",
        how="left_anti"
    )
    .orderBy("mes")
)

In [0]:
display(
    df_coreanos_brasil_mensal
    .filter(F.col("ano") == 2023)
    .groupBy("mes", "mes_num")
    .count()
    .orderBy("mes_num", "mes")
)

In [0]:
display(
    df_brasil
    .filter(
        (F.col("ano") == 2023) &
        (F.col("pais") == "República da Coreia")
    )
    .groupBy("mes")
    .count()
    .orderBy("mes")
)

In [0]:
display(
    df_coreanos_brasil_mensal
    .filter(
        (F.col("ano") == 2023) &
        (F.col("mes_num").isNull())
    )
    .groupBy("mes")
    .count()
)

Salvar tabela

In [0]:
TABELA_GOLD_MENSAL = "chegadas.gold.turismo_bilateral_mensal"

(
    df_gold_mensal.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_GOLD_MENSAL)
)

print("Tabela Gold mensal criada!")

In [0]:
df_mensal_salvo = spark.table(TABELA_GOLD_MENSAL)

print(
    "Registros salvos:",
    df_mensal_salvo.count()
)

display(
    df_mensal_salvo
    .filter(F.col("ano") == 2025)
    .orderBy("mes", "origem")
)